In [86]:
import pandas as pd

def load_data():
    df = pd.read_csv("casey_council_cx_300.csv")
    df["survey_date"] = pd.to_datetime(df["survey_date"])
    return df

df = load_data()


filtered_df = df

monthly = (
    filtered_df
    .set_index("survey_date")
    .groupby("department")["nps_rating"]
    ##.resample("M")
    .resample("ME")
    .mean()
    .reset_index()
)

##print(monthly['nps_rating'].value_counts())


def classify_nps(score):
    if score >= 9:
        return "Promoter"
    elif score >= 7:
        return "Passive"
    else:
        return "Detractor"

df["NPS_Group"] = df["nps_rating"].apply(classify_nps)

print(df["NPS_Group"].value_counts())

ss =  df["NPS_Group"].value_counts().transform(lambda x: x / x.sum() * 100)
print(ss)
counts_table = pd.crosstab(df['department'], df['NPS_Group'])
##print("Counts Table:")
##print(counts_table)
##print("-" * 30)

# # 2. Crosstab with percentages of total
# total_pct_table = pd.crosstab(df['financial_institution'], df['NPS_Group'], normalize='all')
# print("Percentage of Total:")
# print(total_pct_table)
# print(total_pct_table.round(4) * 100) # Multiplying by 100 for better readability as percentages
# print("-" * 30)

# 3. Crosstab with row percentages (sums to 100% horizontally)
##total_pct_table = pd.crosstab(df['financial_institution'], df['NPS_Group'], normalize='columns')
total_pct_table = pd.crosstab(df['department'], df['NPS_Group'], normalize=1)
##print("Percentage of Total:")
##print(total_pct_table)
##print(total_pct_table.round(4) * 100) # Multiplying by 100 for better readability as percentages
print("-" * 30)

counts = pd.crosstab(df['department'], df['NPS_Group'])
pcts = pd.crosstab(df['department'], df['NPS_Group'], normalize='columns').mul(100).round(1)

# Combine them with clear headers
combined = pd.concat([counts, pcts], axis=1, keys=['Count', 'Percentage (%)'])
combined = combined.sort_index(axis=1, level=1)
print(combined)
print("-" * 30)

# 1. Create raw counts and column percentages
counts = pd.crosstab(df['department'], df['NPS_Group'])
pcts = pd.crosstab(df['department'], df['NPS_Group'], normalize='columns') * 100

# 2. Combine into a single string format
# We iterate through the values to create the "Count (Pct%)" format
formatted_df = counts.astype(str) + " (" + pcts.round(1).astype(str) + "%)"

print(formatted_df)


NPS_Group
Detractor    187
Promoter      58
Passive       55
Name: count, dtype: int64
NPS_Group
Detractor    62.333333
Promoter     19.333333
Passive      18.333333
Name: count, dtype: float64
------------------------------
                                        Count Percentage (%)   Count  \
NPS_Group                           Detractor      Detractor Passive   
department                                                             
Community Services                         24           12.8       3   
Customer Experience & Communication        22           11.8       5   
Finance & Rates                            28           15.0       6   
Health & Local Laws                        19           10.2      10   
Infrastructure & Roads                     32           17.1       8   
Leisure & Events                           16            8.6       8   
Parks & Environment                        23           12.3       9   
Planning & Development                     23          

In [124]:
import pandas as pd

def load_data():
    df = pd.read_csv("casey_council_cx_300.csv")
    df["survey_date"] = pd.to_datetime(df["survey_date"])
    return df

df = load_data()

##xtab = pd.crosstab(df['team'],df['department'])
##print(xtab)
##xtab.to_csv("xtab.csv")


# 1. Reshape data so all fruits are in one column linked to Team
##df_melted = df.melt(id_vars=['department'], value_vars=['issue_1', 'issue_2'], value_name='issues')
df_melted = df.melt(id_vars=['department','team'], value_vars=['issue_1', 'issue_2'], value_name='issues')
##print(df_melted)
##df_melted2
# 2. Generate Raw Counts (Rows=Fruit, Cols=Team)
counts = pd.crosstab(df_melted['issues'], df_melted['department'])
count_by_team = pd.crosstab(df_melted['issues'], df_melted['team'])
print(count_by_team)


# 3. Generate Column Percentages (Normalized by Team)
# This shows % of mentions within each Team
##pcts = pd.crosstab(df_melted['Fruit'], df_melted['Team'], normalize='columns').mul(100)
total_people = df['department'].value_counts()
pcts = (counts / total_people * 100)
##print(pcts)
# 4. Format into "Count (Pct%)" strings
##formatted_df = counts.astype(str) + " (" + pcts.round(1).astype(str) + "%)"



print(formatted_df)

##print(df_melted['issues'].value_counts())

##total_people = df['Team'].value_counts()
##pcts = (counts / total_people * 100)

##print(total_people)
##print(pcts)


# 1. Collapse all Fav columns into one single series and drop Nones
all_fruits = df[['issue_1', 'issue_2']].stack()

# 2. Calculate Raw Counts
counts = all_fruits.value_counts()

# 3. Calculate Percentage of Respondents
# Denominator is the total number of people (rows in original df)
num_respondents = len(df)
pcts = (counts / num_respondents * 100).round(1)

# 4. Combine into a single table
summary = pd.DataFrame({
    'Count': counts,
    'Pct of Respondents (%)': pcts
})

# 5. Add Total row
summary.loc['Total'] = summary.sum()

##print(summary)







team                             Aged Care Support  Animal Management  \
issues                                                                  
Aged care coordination                           0                  2   
Animal management response                       0                  0   
Building application process                     1                  0   
Callback reliability                             0                  0   
Communication delays                             2                  1   
Complaint resolution time                        1                  0   
Development application clarity                  0                  0   
Digital service accessibility                    0                  0   
Disability access                                1                  0   
Environmental health follow-up                   0                  0   
Event management                                 0                  0   
Flooding & drainage response                     1 

In [101]:
import pandas as pd

df = pd.DataFrame({
    'department':['AA','AA','AB','AB'],
    'team': ['A', 'A', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange']
})

# 1. Split the string into lists and explode so each fruit has its own row
# This keeps 'department' and 'team' attached to each individual fruit
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')

# 2. Create the crosstab
# Rows: [Department, Fruit]
# Columns: Team
##result = pd.crosstab([df_exploded['department'], df_exploded['Response']], df_exploded['team'])

result = pd.crosstab([df_exploded['department'], df_exploded['Response']], 
                     df_exploded['team'], 
                     margins=True, margins_name="Total Mentions")

# 3. Rename the index for clarity
result.index.names = ['Dept', 'Fruit']

print(result)


team                   A  B  Total Mentions
Dept           Fruit                       
AA             Apple   1  0               1
               Banana  2  0               2
AB             Apple   0  1               1
               Orange  0  2               2
Total Mentions         3  3               6


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'department':['AA','AA', 'BB','BB'],
    'team': ['A', 'C', 'B', 'D'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange']
})

# 1. Explode the multi-response data so each fruit is a row
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')

# 2. Crosstab with Department and Team as the columns (horizontal axis)
# The first list [department, team] creates the nested headers
result = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']],
                     margins=True)

# 3. Clean up labels
result.index.name = 'Fruit'
result.columns.names = ['Dept', 'Team']

print(result)


In [105]:
import pandas as pd

df = pd.DataFrame({
    'department':['AA','AA', 'BB','BB'],
    'team': ['A', 'C', 'B', 'D'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange']
})

# 1. Explode multi-response data
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')

# 2. Create the main Crosstab (Detailed Team view)
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 3. Calculate Department-level Subtotals
# We sum across the second level (Team) for each Department
dept_totals = main_table.groupby(level=0, axis=1).sum()

# Add a 'Subtotal' level to match the main table's column structure
dept_totals.columns = pd.MultiIndex.from_tuples([(d, 'Subtotal') for d in dept_totals.columns])

# 4. Combine and Sort to place subtotals next to their departments
final_table = pd.concat([main_table, dept_totals], axis=1).sort_index(axis=1)

# 5. Add Grand Total at the end
final_table['Grand Total'] = final_table.sum(axis=1)

print(final_table)


         AA             BB             Grand Total
          A  C Subtotal  B  D Subtotal            
Response                                          
Apple     1  0        1  1  0        1           4
Banana    1  1        2  0  0        0           4
Orange    0  0        0  1  1        2           4


/var/folders/ff/4qwktjyd3wj5njzx2mvpy2qc0000gn/T/ipykernel_13987/2142528740.py:17: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dept_totals = main_table.groupby(level=0, axis=1).sum()


In [108]:
import pandas as pd

# 1. Main Crosstab (Only Teams, no subtotals)
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 2. Correct Grand Total (Sum only the raw team columns)
grand_total = main_table.sum(axis=1)

# 3. Calculate Department-level Subtotals
dept_totals = main_table.groupby(level=0, axis=1).sum()
print(dept_totals.columns)
dept_totals.columns = pd.MultiIndex.from_tuples([(d, 'Subtotal') for d in dept_totals.columns])

# 4. Combine Teams and Subtotals
final_table = pd.concat([main_table, dept_totals], axis=1).sort_index(axis=1)

# 5. Add the Corrected Grand Total (Now it won't double count)
final_table[('Grand Total', '')] = grand_total

print(final_table)


Index(['AA', 'BB'], dtype='object', name='department')
         AA             BB             Grand Total
          A  C Subtotal  B  D Subtotal            
Response                                          
Apple     1  0        1  1  0        1           2
Banana    1  1        2  0  0        0           2
Orange    0  0        0  1  1        2           2


/var/folders/ff/4qwktjyd3wj5njzx2mvpy2qc0000gn/T/ipykernel_13987/1015452288.py:10: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dept_totals = main_table.groupby(level=0, axis=1).sum()


In [109]:
import pandas as pd

# 1. Main Crosstab (Raw Teams Only)
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 2. Correct Grand Total Column (Sum before adding subtotals)
grand_total_col = main_table.sum(axis=1)

# 3. Calculate Department Subtotals (Fixed FutureWarning)
# We transpose, group by level 0 (Dept), sum, and transpose back
dept_totals = main_table.T.groupby(level=0).sum().T

# 4. RENAME labels to "Total AA", "Total BB", etc.
# We create a MultiIndex where the second level is "Total " + Dept Name
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])

# 5. Combine Teams and Subtotals & Sort
final_table = pd.concat([main_table, dept_totals], axis=1).sort_index(axis=1)

# 6. Add Corrected Grand Total Column
final_table[('Grand Total', '')] = grand_total_col

# 7. Add Grand Total Row at the bottom
final_table.loc['Grand Total'] = final_table.sum()

print(final_table)


            AA             BB             Grand Total
             A  C Total AA  B  D Total BB            
Response                                             
Apple        1  0        1  1  0        1           2
Banana       1  1        2  0  0        0           2
Orange       0  0        0  1  1        2           2
Grand Total  2  1        3  2  1        3           6


In [110]:
import pandas as pd

# 1. Main Crosstab (Raw Teams Only)
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 2. Correct Grand Total Column (Sum before adding subtotals)
grand_total_col = main_table.sum(axis=1)

# 3. Calculate Department Subtotals (Fixed FutureWarning)
dept_totals = main_table.T.groupby(level=0).sum().T
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])

# 4. Combine Teams and Subtotals
final_table = pd.concat([main_table, dept_totals], axis=1)

# 5. CUSTOM SORT: Sort by Department (Level 0), then by Total vs Team (Level 1)
# We sort level 1 so that strings starting with 'Total' come before Team letters
final_table = final_table.sort_index(axis=1, key=lambda x: x.map(lambda y: (0, y) if 'Total' in str(y) else (1, y)))

# 6. Add Corrected Grand Total Column
final_table[('Grand Total', '')] = grand_total_col

# 7. Add Grand Total Row at the bottom
final_table.loc['Grand Total'] = final_table.sum()

print(final_table)


                  AA             BB       Grand Total
            Total AA  A  C Total BB  B  D            
Response                                             
Apple              1  1  0        1  1  0           2
Banana             2  1  1        0  0  0           2
Orange             0  0  0        2  1  1           2
Grand Total        3  2  1        3  2  1           6


In [114]:
import pandas as pd

# 1. Standard Setup
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 2. Calculate Totals (using the non-deprecated .T.groupby)
dept_totals = main_table.T.groupby(level=0).sum().T
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])

# 3. Combine
final_table = pd.concat([main_table, dept_totals], axis=1)

# 4. MANUALLY REORDER: Define exactly how you want the columns to appear
# We create a list of tuples: (Dept, Sub-header)
new_order = [
    ('AA', 'Total AA'), ('AA', 'A'), ('AA', 'C'),
    ('BB', 'Total BB'), ('BB', 'B'), ('BB', 'D')
]

final_table = final_table[new_order]

# 5. Add Grand Total
final_table[('Grand Total', '')] = main_table.sum(axis=1)
final_table.loc['Grand Total Row'] = final_table.sum()

print(final_table)


                      AA             BB       Grand Total
                Total AA  A  C Total BB  B  D            
Response                                                 
Apple                  1  1  0        1  1  0           2
Banana                 2  1  1        0  0  0           2
Orange                 0  0  0        2  1  1           2
Grand Total Row        3  2  1        3  2  1           6


In [113]:
# Dynamically build the order: Total first, then Teams
dynamic_order = []
for dept in sorted(df['department'].unique()):
    dynamic_order.append((dept, f"Total {dept}")) # Add Total first
    # Add all teams belonging to this dept
    teams = sorted(df[df['department'] == dept]['team'].unique())
    for t in teams:
        dynamic_order.append((dept, t))

final_table = final_table.reindex(columns=dynamic_order)
print(final_table)

                  AA             BB      
            Total AA  A  C Total BB  B  D
Response                                 
Apple              1  1  0        1  1  0
Banana             2  1  1        0  0  0
Orange             0  0  0        2  1  1
Grand Total        3  2  1        3  2  1


In [ ]:
import pandas as pd

# 1. Standard Setup & Data Prep
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])
##print(main_table)
# 2. Add the Grand Total calculation to main_table FIRST
main_table[('Grand Total', '')] = main_table.sum(axis=1)

##print(main_table)

# 3. Calculate Department Subtotals (using fixed syntax)
dept_totals = main_table.drop(columns=[('Grand Total', '')]).T.groupby(level=0).sum().T
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])

# 4. Combine all columns
final_table = pd.concat([main_table, dept_totals], axis=1)

# 5. Dynamically build the order (Including Grand Total at the end)
dynamic_order = []
for dept in sorted(df['department'].unique()):
    dynamic_order.append((dept, f"Total {dept}")) # Total first
    teams = sorted(df[df['department'] == dept]['team'].unique())
    for t in teams:
        dynamic_order.append((dept, t))

# ADD GRAND TOTAL TO THE END OF THE LIST
dynamic_order.append(('Grand Total', ''))

# 6. Apply Reindex and Add Total Row
final_table = final_table.reindex(columns=dynamic_order)
final_table.loc['Grand Total'] = final_table.sum()

print(final_table)


# ... [Your previous code to generate the raw final_table] ...

# 1. Calculate the Column Totals (Denominator)
# This uses the 'Grand Total' row we already created
column_totals = final_table.loc['Grand Total']


# 2. Divide the table by the totals to get percentages
# .div(axis=1) ensures we divide each column by its respective total
pct_table = (final_table.div(column_totals, axis=1) * 100).round(1)



# 3. Optional: Format as strings with '%' symbol
##formatted_pct_table = pct_table.applymap(lambda x: f"{x:.1f}%")
formatted_pct_table = pct_table.map(lambda x: f"{x:.1f}%")

print(formatted_pct_table)

                  AA             BB       Grand Total
            Total AA  A  C Total BB  B  D            
Response                                             
Apple              1  1  0        1  1  0           2
Banana             2  1  1        0  0  0           2
Orange             0  0  0        2  1  1           2
Grand Total        3  2  1        3  2  1           6
                  AA                       BB                 Grand Total
            Total AA       A       C Total BB       B       D            
Response                                                                 
Apple          33.3%   50.0%    0.0%    33.3%   50.0%    0.0%       33.3%
Banana         66.7%   50.0%  100.0%     0.0%    0.0%    0.0%       33.3%
Orange          0.0%    0.0%    0.0%    66.7%   50.0%  100.0%       33.3%
Grand Total   100.0%  100.0%  100.0%   100.0%  100.0%  100.0%      100.0%


In [127]:
# Sample data
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana']
})

# 1. Melt the dataframe to turn Fav_1, Fav_2, Fav_3 into one column
# This keeps 'Team' attached to every individual fruit mention
df_melted = df.melt(id_vars=['Team'], value_vars=['Fav_1', 'Fav_2', 'Fav_3'], value_name='Fruit')
#print(df_melted)
# 2. Create the crosstab (Rows = Fruits, Columns = Teams)
# We dropna() to ignore the 'None' values
result = pd.crosstab(df_melted['Fruit'], df_melted['Team'])
result.loc["Total_raw_count"] = result.sum()
print(result)
# 3. Convert to percentage of respondents
# Total respondents: Team A = 2, Team B = 3
total_respondents = df['Team'].value_counts()
final_table = (result / total_respondents * 100).round(1)

# 4. Add the vertical sum (Total)
final_table.loc['Total'] = final_table.sum()

print(final_table)



Team             A  B
Fruit                
Apple            1  2
Banana           2  1
Grapes           1  0
Guava            1  0
Orange           0  3
Pears            0  1
Total_raw_count  5  7
Team                 A      B
Fruit                        
Apple             50.0   66.7
Banana           100.0   33.3
Grapes            50.0    0.0
Guava             50.0    0.0
Orange             0.0  100.0
Pears              0.0   33.3
Total_raw_count  250.0  233.3
Total            500.0  466.6


In [162]:
import pandas as pd
import numpy as np

# 1. Setup Data
df = pd.DataFrame({
    'department':['AA','AA', 'BB','BB', 'BB'],
    'team': ['A', 'C', 'B', 'D', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange'],
    'survey_date':['2025-01-01','2026-12-01','2025-01-01','2026-12-01','2024-01-01']
})

# 2. Get Respondent Counts (The Denominators)
# Count how many unique people are in each Team
team_counts = df.groupby(['department', 'team']).size()
print(team_counts)

# dept_stats = team_counts.groupby(level='department').agg([
#     ('Min Team Size', 'min'),
#     ('Max Team Size', 'max'),
#     ('Avg Team Size', 'mean')
# ]).reset_index()

# # 3. Round the average for a cleaner look
# dept_stats['Avg Team Size'] = dept_stats['Avg Team Size'].round(1)

dept_stats = team_counts.groupby(level='department').agg([
    ('Number of Teams', 'count'),  # Counts how many teams exist
    ('Min Team Size', 'min'),
    ('Max Team Size', 'max'),
    ('Avg Team Size', 'mean')
]).reset_index()


print(dept_stats)

department  team
AA          A       1
            C       1
BB          B       2
            D       1
dtype: int64
  department  Number of Teams  Min Team Size  Max Team Size  Avg Team Size
0         AA                2              1              1            1.0
1         BB                2              1              2            1.5


In [ ]:
import pandas as pd
import numpy as np

# 1. Setup Data
df = pd.DataFrame({
    'department':['AA','AA', 'BB','BB', 'BB'],
    'team': ['A', 'C', 'B', 'D', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange'],
    'survey_date':['2025-01-01','2026-12-01','2025-01-01','2026-12-01','2024-01-01']
})

# 2. Get Respondent Counts (The Denominators)
# Count how many unique people are in each Team
team_counts = df.groupby(['department', 'team']).size()
print(team_counts)
# Count how many unique people are in each Department
dept_counts = df.groupby('department').size()
print(dept_counts)
# Total unique people
grand_total_respondents = len(df)

# 3. Explode and Create Raw Count Table
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')
main_table = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])

# 4. Calculate Department Totals (Raw Counts)
dept_totals = main_table.T.groupby(level=0).sum().T
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])

# 5. Create Final Raw Table (Adding Grand Total Column)
final_raw = pd.concat([main_table, dept_totals], axis=1)
print(final_raw)
final_raw[('Grand Total', '')] = main_table.sum(axis=1)

# 6. CONVERT TO PERCENTAGE OF RESPONDENTS
# We create a series that matches the columns of final_raw
denominators = {}
for col in final_raw.columns:
    dept, team = col
    ##print(col)
    if dept == 'Grand Total':
        denominators[col] = grand_total_respondents
    elif 'Total' in team:
        denominators[col] = dept_counts[dept]
    else:
        denominators[col] = team_counts[(dept, team)]

# 7. Apply the Division
final_pct = (final_raw.div(pd.Series(denominators), axis=1) * 100).round(1)

# 8. Add the "Total" row (Sum of percentages)
final_pct.loc['Total'] = final_pct.sum()

print(final_pct)


department  team
AA          A       1
            C       1
BB          B       2
            D       1
dtype: int64
department
AA    2
BB    3
dtype: int64
         AA    BB          AA       BB
          A  C  B  D Total AA Total BB
Response                              
Apple     1  0  2  0        1        2
Banana    1  1  0  0        2        0
Orange    0  0  2  1        0        3
             AA            BB              AA       BB Grand Total
              A      C      B      D Total AA Total BB            
Response                                                          
Apple     100.0    0.0  100.0    0.0     50.0     66.7        60.0
Banana    100.0  100.0    0.0    0.0    100.0      0.0        40.0
Orange      0.0    0.0  100.0  100.0      0.0    100.0        60.0
Total     200.0  100.0  200.0  100.0    150.0    166.7       160.0


In [150]:
import pandas as pd
import numpy as np

# 1. Setup Data
df = pd.DataFrame({
    'department':['AA','AA', 'BB','BB', 'BB'],
    'team': ['A', 'C', 'B', 'D', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange']
})

# 2. Get Respondent Counts (The "Of Y" Denominators)
team_resp = df.groupby(['department', 'team']).size()
dept_resp = df.groupby('department').size()
grand_resp = len(df)

# 3. Explode and Create Raw Count Table
df_exploded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')
raw_main = pd.crosstab(df_exploded['Response'], [df_exploded['department'], df_exploded['team']])
print(raw_main.columns)
print(raw_main)
# 4. Add Department Subtotals and Grand Total
dept_totals = raw_main.T.groupby(level=0).sum().T
d_totals = raw_main.T.groupby(level=1).sum()
print(d_totals.T)
print(dept_totals)
print(dept_totals.columns)
dept_totals.columns = pd.MultiIndex.from_tuples([(d, f"Total {d}") for d in dept_totals.columns])
print(dept_totals.columns)
final_raw = pd.concat([raw_main, dept_totals], axis=1)
final_raw[('Grand Total', '')] = raw_main.sum(axis=1)

# 5. Create a Map of Denominators for every column
denom_map = {}
for col in final_raw.columns:
    dept, label = col
    if dept == 'Grand Total':
        denom_map[col] = grand_resp
    elif 'Total' in label:
        denom_map[col] = dept_resp[dept]
    else:
        denom_map[col] = team_resp[(dept, label)]

# 6. Build the Formatted String Table
def format_cell(val, col_name):
    n = denom_map[col_name]
    pct = (val / n * 100)
    return f"{int(val)} of {n} ({pct:.1f}%)"

# Use .map() for element-wise formatting with column context
formatted_df = final_raw.copy()
for col in final_raw.columns:
    formatted_df[col] = final_raw[col].map(lambda x: format_cell(x, col))

# 7. Add the Sum row (Total Average Choices)
# Note: For the summary row, we sum percentages but keep the respondent count
total_pcts = (final_raw.sum() / pd.Series(denom_map) * 100).round(1)
formatted_df.loc['Total'] = [f"{int(final_raw[col].sum())} ({total_pcts[col]}%)" for col in final_raw.columns]

##print(formatted_df)


MultiIndex([('AA', 'A'),
            ('AA', 'C'),
            ('BB', 'B'),
            ('BB', 'D')],
           names=['department', 'team'])
department AA    BB   
team        A  C  B  D
Response              
Apple       1  0  2  0
Banana      1  1  0  0
Orange      0  0  2  1
team      A  B  C  D
Response            
Apple     1  2  0  0
Banana    1  0  1  0
Orange    0  2  0  1
department  AA  BB
Response          
Apple        1   2
Banana       2   0
Orange       0   3
Index(['AA', 'BB'], dtype='object', name='department')
MultiIndex([('AA', 'Total AA'),
            ('BB', 'Total BB')],
           )


In [ ]:
df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange']
})



#####Handling Delimiters: If your data has inconsistent spacing (e.g., "Apple,Banana"), use a regex delimiter: .str.split(r'\s*,\s*') to clean it up during the split.
# 1. Split the strings by the delimiter
# 2. Explode the list into separate rows
# 3. Count the occurrences
counts = df['Response'].str.split(', ').explode().value_counts()

print(counts)


# Create a 0/1 matrix for each unique response
dummies = df['Response'].str.get_dummies(sep=', ')

# Sum the columns to get total counts
response_totals = dummies.sum()

print(response_totals)


# 1 & 2: Split and Explode
# This creates a new row for every individual fruit while keeping the 'Team'
df_expanded = df.assign(Response=df['Response'].str.split(', ')).explode('Response')

# 3. Create the Crosstab
result = pd.crosstab(df_expanded['Response'], df_expanded['Team'])
print(result)

#df_expanded
pct = pd.crosstab(df_expanded['Response'], df_expanded['Team'], normalize='columns') * 100
print(pct)


Response
Apple     2
Banana    2
Orange    2
Name: count, dtype: int64
Apple     2
Banana    2
Orange    2
dtype: int64
Team      A  B
Response      
Apple     1  1
Banana    2  0
Orange    0  2
Team              A          B
Response                      
Apple     33.333333  33.333333
Banana    66.666667   0.000000
Orange     0.000000  66.666667


In [46]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange']
})

# 1. Create a binary matrix (1 if they chose it, 0 if not)
dummies = df['Response'].str.get_dummies(sep=', ')

# 2. Join with the 'Team' column to allow grouping
df_analysis = pd.concat([df['Team'], dummies], axis=1)

# 3. Calculate % of respondents per team
# We group by Team and take the mean (mean of 0s and 1s = percentage)
respondent_pct = df_analysis.groupby('Team').mean().mul(100).round(1)

print(respondent_pct)


total_respondent_pct = dummies.mean().mul(100).round(1)

print(total_respondent_pct)

      Apple  Banana  Orange
Team                       
A      50.0   100.0     0.0
B      66.7     0.0   100.0
Apple     60.0
Banana    40.0
Orange    60.0
dtype: float64


In [45]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange']
})

# 1. Create dummy variables and group by Team
dummies = df['Response'].str.get_dummies(sep=', ')
df_analysis = pd.concat([df['Team'], dummies], axis=1)
final_table = df_analysis.groupby('Team').mean()

# 2. Calculate the Total (percentage of all respondents)
total_row = dummies.mean().to_frame(name='Total').T

# 3. Combine and format
final_table = pd.concat([final_table, total_row])
final_table = final_table.mul(100).round(1)

print(final_table)


       Apple  Banana  Orange
A       50.0   100.0     0.0
B       66.7     0.0   100.0
Total   60.0    40.0    60.0


In [49]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange']
})

# 1. Create binary matrix (1 for selected, 0 for not)
dummies = df['Response'].str.get_dummies(sep=', ')

# 2. Group by Team and calculate the mean (percentage of respondents)
# This initially creates: Rows = Teams, Cols = Responses
team_stats = pd.concat([df['Team'], dummies], axis=1).groupby('Team').mean()

# 3. Transpose (.T) to flip the layout
# This moves Teams to the columns and Responses to the rows
final_table = team_stats.T.mul(100).round(1)
##final_table['Total'] = dummies.mean().mul(100).round(1)

print(final_table)


Team        A      B
Apple    50.0   66.7
Banana  100.0    0.0
Orange    0.0  100.0


In [50]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Response': ['Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange']
})

# 1. Create dummy variables and calculate mean per team
dummies = df['Response'].str.get_dummies(sep=', ')
team_stats = pd.concat([df['Team'], dummies], axis=1).groupby('Team').mean()

# 2. Transpose so Teams are columns and multiply by 100
final_table = team_stats.T.mul(100).round(1)

# 3. Add the 'Total' row at the bottom
# This sums the percentages for each Team (A and B)
final_table.loc['Total'] = final_table.sum()

print(final_table)


Team        A      B
Apple    50.0   66.7
Banana  100.0    0.0
Orange    0.0  100.0
Total   150.0  166.7


In [ ]:
# Sample data
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],å
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana']
})

# 1. Melt the dataframe to turn Fav_1, Fav_2, Fav_3 into one column
# This keeps 'Team' attached to every individual fruit mention
df_melted = df.melt(id_vars=['Team'], value_vars=['Fav_1', 'Fav_2', 'Fav_3'], value_name='Fruit')
#print(df_melted)
# 2. Create the crosstab (Rows = Fruits, Columns = Teams)
# We dropna() to ignore the 'None' values
result = pd.crosstab(df_melted['Fruit'], df_melted['Team'])
result.loc["Total_raw_count"] = result.sum()
print(result)
# 3. Convert to percentage of respondents
# Total respondents: Team A = 2, Team B = 3
total_respondents = df['Team'].value_counts()
final_table = (result / total_respondents * 100).round(1)

# 4. Add the vertical sum (Total)
final_table.loc['Total'] = final_table.sum()

print(final_table)



Team             A  B
Fruit                
Apple            1  2
Banana           2  1
Grapes           1  0
Guava            1  0
Orange           0  3
Pears            0  1
Total_raw_count  5  7
Team                 A      B
Fruit                        
Apple             50.0   66.7
Banana           100.0   33.3
Grapes            50.0    0.0
Guava             50.0    0.0
Orange             0.0  100.0
Pears              0.0   33.3
Total_raw_count  250.0  233.3
Total            500.0  466.6


In [55]:

import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana']
})

# 1. Melt and create the initial crosstab
df_melted = df.melt(id_vars=['Team'], value_vars=['Fav_1', 'Fav_2', 'Fav_3'], value_name='Fruit')
counts = pd.crosstab(df_melted['Fruit'], df_melted['Team'])

# 2. Normalize by total respondents per team
total_resp = df['Team'].value_counts()
pct_table = (counts / total_resp * 100).round(1)

# 3. Sort by total popularity (sum of percentages across teams) and take Top 3
# We create a temporary sum column to sort, then drop it
top_3 = (pct_table.assign(Total_Sum = pct_table.sum(axis=1))
         .sort_values('Total_Sum', ascending=False)
         .head(3)
         .drop(columns='Total_Sum'))

# 4. Add the vertical sum row for the Top 3
top_3.loc['Total'] = top_3.sum()

print(top_3)


Team        A      B
Fruit               
Banana  100.0   33.3
Apple    50.0   66.7
Orange    0.0  100.0
Total   150.0  200.0


In [56]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana']
})

# 1. Collapse all Fav columns into one single series and drop Nones
all_fruits = df[['Fav_1', 'Fav_2', 'Fav_3']].stack()

# 2. Calculate Raw Counts
counts = all_fruits.value_counts()

# 3. Calculate Percentage of Respondents
# Denominator is the total number of people (rows in original df)
num_respondents = len(df)
pcts = (counts / num_respondents * 100).round(1)

# 4. Combine into a single table
summary = pd.DataFrame({
    'Count': counts,
    'Pct of Respondents (%)': pcts
})

# 5. Add Total row
summary.loc['Total'] = summary.sum()

print(summary)


        Count  Pct of Respondents (%)
Apple     3.0                    60.0
Banana    3.0                    60.0
Orange    3.0                    60.0
Grapes    1.0                    20.0
Guava     1.0                    20.0
Pears     1.0                    20.0
Total    12.0                   240.0


In [84]:
import pandas as pd

df = pd.DataFrame({
    'Respo_id':['resp1','resp2','resp3','resp4','resp5'],
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana']
})

# 1. Reshape data so all fruits are in one column linked to Team
df_melted = df.melt(id_vars=['Team'], value_vars=['Fav_1', 'Fav_2', 'Fav_3'], value_name='Fruit')
##print(df_melted)
# 2. Generate Raw Counts (Rows=Fruit, Cols=Team)
counts = pd.crosstab(df_melted['Fruit'], df_melted['Team'])

# 3. Generate Column Percentages (Normalized by Team)
# This shows % of mentions within each Team
##pcts = pd.crosstab(df_melted['Fruit'], df_melted['Team'], normalize='columns').mul(100)
total_people = df['Team'].value_counts()
pcts = (counts / total_people * 100)
##print(pcts)
# 4. Format into "Count (Pct%)" strings
formatted_df = counts.astype(str) + " (" + pcts.round(1).astype(str) + "%)"

print(formatted_df)



##total_people = df['Team'].value_counts()
##pcts = (counts / total_people * 100)

##print(total_people)
##print(pcts)

Team             A           B
Fruit                         
Apple    1 (50.0%)   2 (66.7%)
Banana  2 (100.0%)   1 (33.3%)
Grapes   1 (50.0%)    0 (0.0%)
Guava    1 (50.0%)    0 (0.0%)
Orange    0 (0.0%)  3 (100.0%)
Pears     0 (0.0%)   1 (33.3%)


In [66]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana'],
    'rating':[1, 10, 5, 4, 99]
})

# 1. Replace 99 with NaN so it is excluded from the math
# We use .replace() specifically on the rating column
df['rating_cleaned'] = df['rating'].replace(99, np.nan)

# 2. Group by Team and calculate the mean
# Pandas automatically ignores the NaN (the former 99)
team_averages = df.groupby('Team')['rating_cleaned'].mean().round(2)

print(team_averages)


Team
A    5.5
B    4.5
Name: rating_cleaned, dtype: float64


In [73]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana'],
    'rating':[1, 10, 5, 4, 99]
})

# 1. Clean the 99s first (important so they don't get binned as 'Very Satisfied')
df['rating_clean'] = df['rating'].replace([98, 99],np.nan)

# 2. Define our standard 0-10 to 1-5 bins
# -0.1 ensures '0' is included in the first group
bins = [-0.1, 2, 4, 6, 8, 10]

# 3. Create the Numeric 1-5 Column
numeric_labels = [1, 2, 3, 4, 5]
df['rating_1to5'] = pd.cut(df['rating_clean'], bins=bins, labels=numeric_labels)

# 4. Create the Sentiment Column
sentiment_labels = [
    'Very Dissatisfied', 
    'Dissatisfied', 
    'Neutral', 
    'Satisfied', 
    'Very Satisfied',
    
]
df['sentiment'] = pd.cut(df['rating_clean'], bins=bins, labels=sentiment_labels)
df['sentiment'] = df['sentiment'].cat.add_categories(['Not Applicable']).fillna('Not Applicable')


# Clean up: Convert numeric column from 'category' type to Integer (handles NaNs as <NA>)
df['rating_1to5'] = df['rating_1to5'].astype(float).astype('Int64')

print(df[['Team', 'rating', 'rating_1to5', 'sentiment']])


  Team  rating  rating_1to5          sentiment
0    A       1            1  Very Dissatisfied
1    A      10            5     Very Satisfied
2    B       5            3            Neutral
3    B       4            2       Dissatisfied
4    B      99         <NA>     Not Applicable


In [79]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'Fav_1': ['Apple', 'Banana', 'Orange', 'Orange', 'Apple'],
    'Fav_2': ['Banana', 'Guava', 'Apple', None, 'Orange'],
    'Fav_3':['Grapes', None, 'Pears', None, 'Banana'],
    'rating':[1, 10, 5, 4, 99],
    'budget':[1000, 2000, 900, 300, 500]
})

# 1. Group by Team and sum the budget
team_budget = df.groupby('Team')['budget'].sum().reset_index()

# 2. Optional: Add a Total row at the bottom
total_sum = team_budget['budget'].sum()
team_budget.loc[len(team_budget)] = ['Total', total_sum]

print(team_budget)

budget_summary = df.groupby('Team')['budget'].agg(['sum', 'mean']).round(2)
budget_summary


    Team  budget
0      A    3000
1      B    1700
2  Total    4700


,sum,mean
Team,,
A,3000,1500.00
B,1700,566.67


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'budget':[1000, 2000, 900, 300, 500]
})

# 1. Group by Team and sum the budget
budget_summary = df.groupby('Team')['budget'].sum().reset_index()

# 2. Calculate Percentage of Total
total_pool = budget_summary['budget'].sum()
budget_summary['Percentage (%)'] = (budget_summary['budget'] / total_pool * 100).round(1)

# 3. Add a Total row for clarity
total_row = pd.DataFrame({'Team': ['Total'], 'budget': [total_pool], 'Percentage (%)': [100.0]})
budget_summary = pd.concat([budget_summary, total_row], ignore_index=True)

print(budget_summary)


In [ ]:
df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'nps_rating':[10,8,10,1,10],
    'budget':[1000, 2000, 900, 300, 500]
})



def calculate_nps(ratings):
    # Promoters (9-10), Passives (7-8), Detractors (0-6)
    promoters = (ratings >= 9).sum()
    detractors = (ratings <= 6).sum()
    total = len(ratings)
    
    return ((promoters - detractors) / total) * 100

# Group by Team and apply the calculation
nps_by_team = df.groupby('Team')['nps_rating'].apply(calculate_nps).reset_index()
print(nps_by_team)
##nps_by_team.columns = ['Team', 'NPS_Score']
##nps_by_team.columns = ['Team', 'nps_rating:NPS_Score']
nps_by_team.rename(columns={'nps_rating': 'NPS_Score'}, inplace=True)

print(nps_by_team)
overall_nps = calculate_nps(df['nps_rating'])

print(f"Overall NPS Score: {overall_nps}")


def nps_group(score):
    if score <= 6:
        return "Detractor"
    elif score <= 8:
        return "Passive"
    else:
        return "Promoter"

df["NPS_Group"] = df["nps_rating"].apply(nps_group)

# nps_by_team = (
#     df.groupby("Team")
#     .apply(lambda x:
#         (x["NPS_Group"].eq("Promoter").mean()
#          - x["NPS_Group"].eq("Detractor").mean()) * 100
#     )
#     .reset_index(name="NPS")
# )

nps_by_team = (
    df.assign(
        promoter=lambda d: d["NPS_Group"].eq("Promoter"),
        detractor=lambda d: d["NPS_Group"].eq("Detractor")
    )
    .groupby("Team")
    .agg(
        promoters=("promoter", "mean"),
        detractors=("detractor", "mean")
    )
)

nps_by_team["NPS"] = (nps_by_team["promoters"]
                      - nps_by_team["detractors"]) * 100

nps_by_team = nps_by_team.reset_index()[["Team","NPS"]]


print(nps_by_team)

summary = (
    df.groupby(["Team","NPS_Group"])
    .size()
    .unstack(fill_value=0)
)

summary["Total"] = summary.sum(axis=1)

summary["NPS"] = (
    (summary["Promoter"]/summary["Total"]
     - summary["Detractor"]/summary["Total"]) * 100
)

summary.reset_index()

  Team  nps_rating
0    A   50.000000
1    B   33.333333
  Team  nps_rating
0    A   50.000000
1    B   33.333333
Overall NPS Score: 40.0
  Team        NPS
0    A  50.000000
1    B  33.333333


NPS_Group,Team,Detractor,Passive,Promoter,Total,NPS
0,A,0,1,1,2,50.000000
1,B,1,0,2,3,33.333333


In [178]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'nps_rating': [10, 8, 10, 1, 10],
    'budget': [1000, 2000, 900, 300, 500]
})

# NPS group classification
df["nps_group"] = df["nps_rating"].apply(
    lambda x: "Promoter" if x >= 9 else ("Passive" if x >= 7 else "Detractor")
)

# NPS score per team
nps_by_team = (
    df.groupby("Team").apply(
        lambda g: round(
            ((g["nps_group"] == "Promoter").sum() - (g["nps_group"] == "Detractor").sum()) / len(g) * 100, 1
        ),
        include_groups=False
    )
    .reset_index(name="NPS Score")
)

print(nps_by_team)


  Team  NPS Score
0    A       50.0
1    B       33.3


In [ ]:
df1 = pd.DataFrame({
    'Team': ['A', 'A', 'B', 'B', 'B'],
    'nps_rating': [10, 8, 10, 1, 10],
    'postcode': [3806, 3978, 3804, 3174, 3000]
})


df2 = pd.DataFrame({
    'postcode': ["3156", "3177", "3802", "3803", "3804", "3805", "3806", "3807", "3912", "3975", "3976", "3977", "3978", "3980"]
})



In [2]:
import pandas as pd

def load_data():
    df = pd.read_csv("casey_council_cx_300.csv")
    df["survey_date"] = pd.to_datetime(df["survey_date"])
    return df

df1 = load_data()

df2 = pd.read_csv(
    "suburbs-w-postcodes.csv", 
    sep=';', 
    ##usecols=['Postcode'], 
    encoding='utf-8-sig'
)

# 1. Ensure data types match
df1['postcode'] = df1['postcode'].astype(int)
df2['Postcode'] = df2['Postcode'].astype(int)

# 2. Filter for missing postcodes and select only the 'postcode' column
# We add .unique() to get a clean list of just the numbers
missing_postcodes = df1.loc[~df1['postcode'].isin(df2['Postcode']), 'postcode'].unique()

print("Postcodes in df1 that are NOT in the CSV list:")
print(missing_postcodes)
# with open("suburbs-w-postcodes.csv", 'r') as f:
#     for i in range(5):
#         print(f"Line {i}: {f.readline()}")


# header_test = pd.read_csv("suburbs-w-postcodes.csv", nrows=0)
# print(f"Columns found: {header_test.columns.tolist()}")

Postcodes in df1 that are NOT in the CSV list:
[3808 3809 3810 3174 3979 3158]


In [ ]:
import chardet

def inspect_encoding(path, sample_size=50000):
    with open(path, 'rb') as f:
        raw = f.read(sample_size)

    # BOM check
    boms = {
        b'\xff\xfe\x00\x00': 'UTF-32-LE',
        b'\x00\x00\xfe\xff': 'UTF-32-BE',
        b'\xef\xbb\xbf':     'UTF-8-SIG',
        b'\xff\xfe':         'UTF-16-LE',
        b'\xfe\xff':         'UTF-16-BE',
    }
    bom_found = next((name for bom, name in boms.items() if raw.startswith(bom)), None)

    # chardet detection
    detected  = chardet.detect(raw)

    # Try decoding with common encodings to verify
    common = ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252', 'utf-16', 'ascii']
    valid = []
    for enc in common:
        try:
            raw.decode(enc)
            valid.append(enc)
        except (UnicodeDecodeError, LookupError):
            valid.append(f"{enc} ✗")

    print(f"{'─'*40}")
    print(f" BOM detected    : {bom_found or 'None'}")
    print(f" chardet guess   : {detected['encoding']}")
    print(f" Confidence      : {detected['confidence']*100:.1f}%")
    print(f" Language hint   : {detected['language'] or 'N/A'}")
    print(f"{'─'*40}")
    print(f" Compatible encodings:")
    for enc in valid:
        print(f"   {'✓' if '✗' not in enc else '✗'}  {enc}")
    print(f"{'─'*40}")



In [5]:
import pandas as pd

df = pd.read_csv('cx_survey1_rows.csv')

# Replace one value
df['suburb'] = df['suburb'].replace('Cardinia', 'Clyde')

df.to_csv('cx_survey1_rows.csv', index=False)

In [31]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A','A', 'A', 'B', 'B', 'B','C'],
    'Favorites': ['Banana, Apple','Apple, Banana', 'Banana', 'Apple, Orange', 'Orange', 'Apple, Orange','Apple, Banana, Orange']
})



def grouped_count_table(df, col):
    df = df.copy()

    # ✅ Sort items within each cell so "Banana, Apple" == "Apple, Banana"
    df[col] = df[col].apply(
        lambda x: ', '.join(sorted([i.strip() for i in x.split(',')]))
    )
    

    df['n'] = df[col].str.split(',').str.len()
    counts = df.groupby(['n', col]).size().reset_index(name='count')
    ##print(df)
    print(counts)
    rows = []
    for n, group in counts.groupby('n'):
        label = f"{n} response{'s' if n > 1 else ''}"
        rows.append({col: label, 'count': ''})
        for _, row in group.iterrows():
            rows.append({col: f"  {row[col]}", 'count': row['count']})

    return pd.DataFrame(rows).set_index(col)

print(grouped_count_table(df, 'Favorites').to_string())

   n              Favorites  count
0  1                 Banana      1
1  1                 Orange      1
2  2          Apple, Banana      2
3  2          Apple, Orange      2
4  3  Apple, Banana, Orange      1
                        count
Favorites                    
1 response                   
  Banana                    1
  Orange                    1
2 responses                  
  Apple, Banana             2
  Apple, Orange             2
3 responses                  
  Apple, Banana, Orange     1


In [11]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Favorites': ['Banana, Apple', 'Apple, Banana', 'Banana', 
                  'Apple, Orange', 'Orange', 'Apple, Orange', 'Apple, Banana, Orange']
})

def grouped_count_table(df, col):
    df = df.copy()
    
    # Normalise order: "Banana, Apple" → "Apple, Banana"
    df[col] = df[col].apply(
        lambda x: ', '.join(sorted([i.strip() for i in x.split(',')]))
    )
    
    df['n'] = df[col].str.split(',').str.len()
    counts = df.groupby(['n', col]).size().reset_index(name='count')

    rows = []
    for n, group in counts.groupby('n'):
        label = f"{n} response{'s' if n > 1 else ''}"
        group_total = group['count'].sum()   # ✅ sum of counts in this group
        rows.append({col: label, 'count': group_total})   # ✅ show total on header row
        for _, row in group.iterrows():
            rows.append({col: f"  {row[col]}", 'count': row['count']})

    return pd.DataFrame(rows).set_index(col)

print(grouped_count_table(df, 'Favorites').to_string())


                         count
Favorites                     
1 response                   2
  Banana                     1
  Orange                     1
2 responses                  4
  Apple, Banana              2
  Apple, Orange              2
3 responses                  1
  Apple, Banana, Orange      1


In [14]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Favorites': ['Banana, Apple', 'Apple, Banana', 'Banana', 
                  'Apple, Orange', 'Orange', 'Apple, Orange', 'Apple, Banana, Orange']
})

# 1. Count the number of items in each row (split by comma)
# 2. Get the value_counts of those lengths
# 3. Sort the index so it goes 1, 2, 3...
print(df['Favorites'].str.split(',').str.len().value_counts())
response_counts = df['Favorites'].str.split(',').str.len().value_counts().sort_index()
print(response_counts)
# 4. Format into a clean table
result = pd.DataFrame({
    'Responses': [f"{int(i)} response{'s' if i > 1 else ''}" for i in response_counts.index],
    'count': response_counts.values
})

print(result)

Favorites
2    4
1    2
3    1
Name: count, dtype: int64
Favorites
1    2
2    4
3    1
Name: count, dtype: int64
     Responses  count
0   1 response      2
1  2 responses      4
2  3 responses      1


In [18]:
import pandas as pd



df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Fav_1': ['Banana', 'Apple', 'Banana', 'Apple', 'Orange', 'Apple', 'Apple'],              
    'Fav_2': ['Apple', 'Banana', None, 'Orange', None, 'Orange', 'Banana'],
    'Fav_3': [None, None, None, None, None, None, 'Orange']                                       
})

# 1. Select only the columns that start with 'Fav_'
# 2. Use count(axis=1) to count non-null values in each row
# 3. Use value_counts() to summarize how many people gave 1, 2, or 3 responses
response_counts = df.filter(like='Fav_').count(axis=1).value_counts().sort_index()
print(df.filter(like='Fav_').count(axis=1).value_counts())
# 4. Format into the requested table
result = pd.DataFrame({
    'Responses': [f"{int(i)} response{'s' if i > 1 else ''}" for i in response_counts.index],
    'count': response_counts.values
})

print(result.to_string(index=False))

2    4
1    2
3    1
Name: count, dtype: int64
  Responses  count
 1 response      2
2 responses      4
3 responses      1


In [21]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Fav_1': ['Banana', 'Apple', 'Banana', 'Apple', 'Orange', 'Apple', 'Apple'],              
    'Fav_2': ['Apple', 'Banana', None, 'Orange', None, 'Orange', 'Banana'],
    'Fav_3': [None, None, None, None, None, None, 'Orange']                                       
})

# 1. Identify the columns starting with 'Fav_'
fav_cols = df.filter(like='Fav_')

# 2. Join the non-null values for each row into a single string
# .dropna() removes the None/NaN values before joining
df['Favorites'] = fav_cols.apply(lambda row: ', '.join(row.dropna()), axis=1)
print(df['Favorites'].unique())

# 3. Optional: Create the summary table you asked for previously
response_counts = df['Favorites'].str.split(',').str.len().value_counts().sort_index()
print(response_counts)
summary = pd.DataFrame({
    'Responses': [f"{int(i)} response{'s' if i > 1 else ''}" for i in response_counts.index],
    'count': response_counts.values
})

# 3. Print results without the index
print("--- Updated Dataframe ---")
print(df[['Team', 'Favorites']].to_string(index=False))

print("\n--- Summary Table ---")
print(summary.to_string(index=False))

['Banana, Apple' 'Apple, Banana' 'Banana' 'Apple, Orange' 'Orange'
 'Apple, Banana, Orange']
Favorites
1    2
2    4
3    1
Name: count, dtype: int64
--- Updated Dataframe ---
Team             Favorites
   A         Banana, Apple
   A         Apple, Banana
   A                Banana
   B         Apple, Orange
   B                Orange
   B         Apple, Orange
   C Apple, Banana, Orange

--- Summary Table ---
  Responses  count
 1 response      2
2 responses      4
3 responses      1


In [26]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Fav_1': ['Banana', 'Apple', 'Banana', 'Apple', 'Orange', 'Apple', 'Apple'],              
    'Fav_2': ['Apple', 'Banana', "None", 'Orange', "Don't know", 'Orange', 'Banana'],
    'Fav_3': ["Not Applicable", None, None, None, None, None, 'Orange']                                       
})

# 1. Identify the columns starting with 'Fav_'
fav_cols = df.filter(like='Fav_')
print(fav_cols)

# 2. Join the non-null values for each row into a single string
# .dropna() removes the None/NaN values before joining
df['Favorites'] = fav_cols.apply(lambda row: ', '.join(row.dropna()), axis=1)
##print(df['Favorites'].unique())

# 3. Optional: Create the summary table you asked for previously
response_counts = df['Favorites'].str.split(',').str.len().value_counts().sort_index()
print(response_counts)
summary = pd.DataFrame({
    'Responses': [f"{int(i)} response{'s' if i > 1 else ''}" for i in response_counts.index],
    'count': response_counts.values
})

# 3. Print results without the index
print("--- Updated Dataframe ---")
print(df[['Team', 'Favorites']].to_string(index=False))

print("\n--- Summary Table ---")
print(summary.to_string(index=False))

    Fav_1       Fav_2           Fav_3
0  Banana       Apple  Not Applicable
1   Apple      Banana            None
2  Banana        None            None
3   Apple      Orange            None
4  Orange  Don't know            None
5   Apple      Orange            None
6   Apple      Banana          Orange
Favorites
2    5
3    2
Name: count, dtype: int64
--- Updated Dataframe ---
Team                     Favorites
   A Banana, Apple, Not Applicable
   A                 Apple, Banana
   A                  Banana, None
   B                 Apple, Orange
   B            Orange, Don't know
   B                 Apple, Orange
   C         Apple, Banana, Orange

--- Summary Table ---
  Responses  count
2 responses      5
3 responses      2


In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'Fav_1': ['Banana', 'Apple', 'Banana', 'Apple', 'Orange', 'Apple', 'Apple'],              
    'Fav_2': ['Apple', 'Banana', "None", 'Orange', "Don't know", 'Orange', 'Banana'],
    'Fav_3': ["Not Applicable", None, None, None, None, None, 'Orange']                                       
})

# 1. Define the exclusion list
exclude = ["None", "Don't know", "Not Applicable"]

# 2. Identify the Fav columns
fav_cols = df.filter(like='Fav_')
print(fav_cols)

# 3. Create a helper function to filter each row
# This drops actual nulls AND ignores the strings in your exclude list
def get_valid_favs(row):
    print(row)
    return [str(val) for val in row.dropna() if str(val) not in exclude]

# Apply the helper function to get a list of valid responses per row
temp_lists = fav_cols.apply(get_valid_favs, axis=1)
print(temp_lists)
# 4. Create the 'Favorites' string column
df['Favorites'] = temp_lists.str.join(', ')

# 5. Create the Summary Table based on the count of the valid lists
response_counts = temp_lists.str.len().value_counts().sort_index()

summary = pd.DataFrame({
    'Responses': [f"{int(i)} response{'s' if i > 1 else ''}" for i in response_counts.index],
    'count': response_counts.values
})

# 6. Print results without the index
print("--- Updated Dataframe ---")
print(df[['Team', 'Favorites']].to_string(index=False))

print("\n--- Summary Table ---")
print(summary.to_string(index=False))

In [32]:
import uuid

# Generate a random UUID v4 object
new_id = uuid.uuid4()

print(new_id)            # e.g., 'fbd204a7-318e-4dd3-86e0-e6d524fc3f98'
print(type(new_id))      # <class 'uuid.UUID'>

21492cef-bf66-4f05-ab63-6b915ed3e82b
<class 'uuid.UUID'>


In [98]:
import pandas as pd
import os

# Load your files
df = pd.read_csv("api_vic_fuel_prices.csv")
df_brand_name = pd.read_csv("api_vic_fuel_brands.csv")

# Perform the merge (SQL LEFT JOIN equivalent)
# We use 'brand_id' from your fuel data and 'id' from the brand list
df = pd.merge(
    df, 
    df_brand_name[['id', 'name']], 
    left_on='brand_id', 
    right_on='id', 
    how='left'
)

# Cleanup: Drop the redundant 'id' column created by the merge
df.drop(columns=['id'], inplace=True)

# Optional: Handle any brands not found in the reference file
##df['brand_name'] = df['brand_name'].fillna('Independent')
df.rename(columns={'name': 'brand_name'}, inplace=True)
# Count unique stations (excluding duplicates)
##unique_stations = df['id'].nunique()

##print(f"Total Unique Stations: {unique_stations}")
##print(df[['fuelStation_name', 'brandId', 'brand_name']].head())
##df.to_csv("vic_fuel_prices_with_brands.csv", index=False)

##print("✅ Merged file saved as: vic_fuel_prices_with_brands.csv")

# ── Split & save one CSV per fuel type ───────────────────────────────────────
fuel_types = df["fuel_type"].unique()

# for fuel in fuel_types:
#     fuel_df = df[df["fuel_type"] == fuel].reset_index(drop=True)
#     filename = f"vic_fuel_{fuel}.csv"
#     ##fuel_df.to_csv(filename, index=False, encoding="utf-8-sig")
#     fuel_df.to_csv(filename, index=False, encoding="utf-8")
#     print(f"✅ Saved: {filename}  ({len(fuel_df)} row{'s' if len(fuel_df) > 1 else ''})")


# output_dir = "fuel_exports"
# os.makedirs(output_dir, exist_ok=True)       # create folder if not exists

# for fuel in df["fuel_type"].unique():
#     fuel_df = df[df["fuel_type"] == fuel].reset_index(drop=True)
#     path = os.path.join(output_dir, f"fuel_{fuel}.csv")
#     fuel_df.to_csv(path, index=False, encoding="utf-8-sig")
#     print(f"✅ {path}  ({len(fuel_df)} rows)")

cheapest = (
    df[df["is_available"] == True]          # only available fuel
    .sort_values("price")                    # sort ascending
    .groupby("fuel_type")                    # group by fuel type
    .first()                               # take cheapest (first row)
    .reset_index()
)[["fuel_type", "brand_id", "station_name", "address", "price"]]

print(cheapest.to_string(index=False))

top3_cheapest = (
    df[df["is_available"] == True]          # only available fuel
    .sort_values("price", ascending=True)   # cheapest first
    .groupby("fuel_type")
    .head(3)                                # top 3 per group
    .reset_index(drop=True)
)

print(top3_cheapest[["fuel_type","station_name","price","is_available"]].to_string(index=False))

top3_cheapest["rank"] = (
    top3_cheapest
    .groupby("fuel_type")["price"]
    .rank(method="min", ascending=True)
    .astype(int)
)

# Add rank emoji
top3_cheapest["rank_label"] = top3_cheapest["rank"].map({1:"🥇",2:"🥈",3:"🥉"})

result = top3_cheapest[[
    "fuel_type","rank_label","station_name","street","price","price_updatedAt","postcode","suburb"
]].sort_values(["fuel_type","price"])

print(result.to_string(index=False))

fuel_type           brand_id       station_name                                  address  price
      DSL a03Ol00000OhlFZIAZ APCO Kangaroo Flat   35-37 High Street, Kangaroo Flat, 3555  159.3
      E10 a03Ol00000NiMxKIAV      LIBERTY EPSOM         213 MIDLAND HIGHWAY, EPSOM, 3551  147.3
      E85 a03Ol00000NiMxhIAF       ASTRON CORIO  160-164 Bacchus Marsh Road, CORIO, 3214  199.9
      LPG a03Ol00000NiMxvIAF    Metro Frankston      86 Cranbourne Road, FRANKSTON, 3199   76.7
      P95 a03Ol00000OhlFZIAZ  APCO Mount Duneed 740-750 Torquay Road, Mount Duneed, 3217  157.7
      P98 a03Ol00000OhlFZIAZ     APCO Grovedale    133-143 Torquay Road, Grovedale, 3216  166.7
     PDSL a03Ol00000NiMyAIAV  mobil white hills     588 Napier Street, WHITE HILLS, 3550  159.3
      U91 a03Ol00000OhlFZIAZ       APCO Highton     250 South Valley Road, Highton, 3216  149.7
fuel_type                 station_name  price  is_available
      LPG              Metro Frankston   76.7          True
      LPG       

In [ ]:
import pandas as pd
import os

# Load your files
df = pd.read_csv("api_vic_fuel_prices.csv")
df_brand_name = pd.read_csv("api_vic_fuel_brands.csv")

# Perform the merge (SQL LEFT JOIN equivalent)
# We use 'brand_id' from your fuel data and 'id' from the brand list
df = pd.merge(
    df, 
    df_brand_name[['id', 'name']], 
    left_on='brand_id', 
    right_on='id', 
    how='left'
)

# Cleanup: Drop the redundant 'id' column created by the merge
df.drop(columns=['id'], inplace=True)

# Optional: Handle any brands not found in the reference file
##df['brand_name'] = df['brand_name'].fillna('Independent')
df.rename(columns={'name': 'brand_name'}, inplace=True)
# Count unique stations (excluding duplicates)
##unique_stations = df['id'].nunique()

##print(f"Total Unique Stations: {unique_stations}")
##print(df[['fuelStation_name', 'brandId', 'brand_name']].head())
##df.to_csv("vic_fuel_prices_with_brands.csv", index=False)

##print("✅ Merged file saved as: vic_fuel_prices_with_brands.csv")

# ── Split & save one CSV per fuel type ───────────────────────────────────────
fuel_types = df["fuel_type"].unique()

# for fuel in fuel_types:
#     fuel_df = df[df["fuel_type"] == fuel].reset_index(drop=True)
#     filename = f"vic_fuel_{fuel}.csv"
#     ##fuel_df.to_csv(filename, index=False, encoding="utf-8-sig")
#     fuel_df.to_csv(filename, index=False, encoding="utf-8")
#     print(f"✅ Saved: {filename}  ({len(fuel_df)} row{'s' if len(fuel_df) > 1 else ''})")


# output_dir = "fuel_exports"
# os.makedirs(output_dir, exist_ok=True)       # create folder if not exists

# for fuel in df["fuel_type"].unique():
#     fuel_df = df[df["fuel_type"] == fuel].reset_index(drop=True)
#     path = os.path.join(output_dir, f"fuel_{fuel}.csv")
#     fuel_df.to_csv(path, index=False, encoding="utf-8-sig")
#     print(f"✅ {path}  ({len(fuel_df)} rows)")

# cheapest = (
#     df[df["is_available"] == True]          # only available fuel
#     .sort_values("price")                    # sort ascending
#     .groupby("fuel_type")                    # group by fuel type
#     .first()                               # take cheapest (first row)
#     .reset_index()
# )[["fuel_type", "brand_id", "station_name", "address", "price"]]

##print(cheapest.to_string(index=False))

print(df.columns)

target_brands = ['APOLLO FUEL', 'SHELL', 'BP']
by_fuel_type = df[(df["is_available"] == True) & (df['brand_name'].isin(target_brands))].groupby(["fuel_type","suburb","station_name"]).agg(
    high_price=("price","max"),
    low_price=("price","min"),
    count=("fuel_type",'size') 
)
##print(by_fuel_type)


by_suburb = df[
    (df["is_available"] == True) & 
    ##(df['brand_name'].isin(['BP'])) & 
    (df['fuel_type'] =='P98') &
    (df['suburb']=='BERWICK')
    ].groupby(["suburb"
               ,"fuel_type"
               ##,"station_name"
               ]
               ).agg(
    high_price=("price","max"),
    low_price=("price","min"),
    count=("fuel_type",'size') 
)
##print(by_suburb)



##print(df['brand_name'].value_counts(ascending=True))
##print(df['brand_name'].value_counts().sort_index())


# 1. Define your filters
filtered_df = df[
    (df["is_available"] == True) & 
    (df['fuel_type'] == 'P95') &
    (df['suburb'] == 'BERWICK')
]

# 2. Sort by price (ascending), then group and take the first record
##by_suburb2 = filtered_df.sort_values("price").groupby(["suburb", "fuel_type"]).first()

# 3. Select only the columns you want to see
##print(by_suburb2[["station_name", "price", "brand_name"]])


top3_cheapest = (
    filtered_df          # only available fuel
    .sort_values("price", ascending=True)   # cheapest first
    .groupby("station_name")
    .head(3)                              # top 3 per group
    .reset_index(drop=True)
)

# print(top3_cheapest[["fuel_type","station_name","price","is_available"]].to_string(index=False))

##print(top3_cheapest[['station_name','price']])
# top3_cheapest["rank"] = (
#     top3_cheapest
#     .groupby("fuel_type")["price"]
#     .rank(method="min", ascending=True)
#     .astype(int)
# )



cheapest = (
    filtered_df          # only available fuel
    .sort_values("price", ascending=True)   # cheapest first
    .groupby("station_name")
    .head(3)                           # top 3 per group
    .reset_index(drop=True)
)
cheapest_1 = (
    filtered_df
    .sort_values("price", ascending=True)
    .head(1)  # Change 3 to 1
)
print(cheapest_1[['address','station_name','price']])



# This finds the single (1) row with the smallest "price"
cheapest_1 = filtered_df.nsmallest(1, "price")

print(cheapest_1)

Index(['station_id', 'station_name', 'brand_id', 'address', 'contact_phone',
       'latitude', 'longitude', 'station_updatedAt', 'fuel_type', 'price',
       'is_available', 'price_updatedAt', 'address_len', 'street', 'suburb',
       'postcode', 'brand_name'],
      dtype='object')
                           address      station_name  price
5998  47 Clyde Road, Berwick, 3806  EG Ampol Berwick  236.9
                                        station_id      station_name  \
5998  VkESR5HAz2q6Yy2AZbu8X0NpgnjvfE3w+S9E9LfBRlQ=  EG Ampol Berwick   

                brand_id                       address contact_phone  \
5998  a03Ol00000NiMxoIAF  47 Clyde Road, Berwick, 3806  03 5219 3636   

       latitude   longitude         station_updatedAt fuel_type  price  \
5998 -38.038902  145.342751  2026-03-03T19:06:15.467Z       P95  236.9   

      is_available           price_updatedAt  address_len         street  \
5998          True  2026-03-03T19:06:15.467Z            3  47 Clyde Road   

   

In [ ]:

import pandas as pd # Explicitly import pandas here
import json

df = pd.read_csv("api_vic_fuel_prices.csv")
print(df['station_id'].nunique())
fil_sub = ['BERWICK','CRANBOURNE','CRANBOURNE NORTH','NARRE WARREN','NARRE WARREN NORTH']
filtered_df = df[
    (df["is_available"] == True) &
    (df['suburb'].isin(fil_sub))
]

# Aggregate fuel types and prices for each station into a single string for hover
station_fuel_details = filtered_df.groupby(['station_id', 'station_name', 'address', 'latitude', 'longitude', 'postcode']).apply(
    lambda x: '<br>'.join([f"{row['fuel_type']}: {row['price']}" for index, row in x.iterrows()])
).reset_index(name='all_fuel_prices_available')


print(station_fuel_details.columns)
print(station_fuel_details[['station_id','all_fuel_prices_available']])

# # This aggregated DataFrame (station_fuel_details) now has one row per unique station
# station_plot_df = station_fuel_details.copy()

# # Diagnostic prints to verify dataframes
# print(f"Number of rows in original filtered_df: {len(filtered_df)}")
# print(f"Number of unique stations in filtered_df: {filtered_df['station_id'].nunique()}")
# print(f"Number of rows in station_plot_df (one per station): {len(station_plot_df)}")
# print("Sample of station_plot_df:")
# print(station_plot_df[['station_name', 'postcode', 'all_fuel_prices_available']].head())



In [ ]:
import pandas as pd

import json

df = pd.read_csv('api_vic_fuel_prices.csv')

# # Create a dataframe for each fuel type
# df_U91  = df[df['fuel_type'] == 'U91']
# df_P98  = df[df['fuel_type'] == 'P98']
# df_DSL  = df[df['fuel_type'] == 'DSL']
# df_PDSL = df[df['fuel_type'] == 'PDSL']
# df_LPG  = df[df['fuel_type'] == 'LPG']
# df_E10  = df[df['fuel_type'] == 'E10']
# df_E85  = df[df['fuel_type'] == 'E85']
# df_P95  = df[df['fuel_type'] == 'P95']
# df_B20  = df[df['fuel_type'] == 'B20']
# df_LNG  = df[df['fuel_type'] == 'LNG']


##print(df.groupby('fuel_type').size())

ss = df.groupby('fuel_type').agg({
    'station_name':['size'],
    'price':['min','max','mean']
})

# Or more efficiently, use a dictionary
dfs = {fuel: df[df['fuel_type'] == fuel] for fuel in df['fuel_type'].unique()}

print(type(dfs))
##print(dfs)

# Option 1: Convert each DataFrame value to records
json_ready = {key: df.to_dict(orient='records') for key, df in dfs.items()}
print(json.dumps(json_ready, indent=2))


# Option 2: Convert each DataFrame value to JSON string
# json_ready = {key: json.loads(df.to_json(orient='records')) for key, df in dfs.items()}
# print(json.dumps(json_ready, indent=2))

#
##print(ss)

st = (
    df[df["is_available"] == True]
    .dropna(subset=['price']) # Explicitly removes NaN/Null prices
    .groupby('fuel_type')
    .agg({
        'station_name': ['size'],
        'price': ['min', 'max', 'mean']
    })
)

##print(type(st))


sx = df.groupby('fuel_type').agg({
    'station_name': ['size'], # Counts everything
    'price': [
        ('min', lambda x: x[df.loc[x.index, "is_available"] == True].min()),
        ('max', lambda x: x[df.loc[x.index, "is_available"] == True].max()),
        ('mean', lambda x: x[df.loc[x.index, "is_available"] == True].mean())
    ]
})

##print(sx)

df['price_filtered'] = df['price'].where((df['is_available'] == True) & (df['price'].notnull()))

##df['price_filtered'] = pd.to_numeric(df['price_filtered'], errors='coerce')

sy = (df[df["is_available"] == True]
     .groupby('fuel_type').agg({
    'station_name': [('cc', 'size'),
                     ('counts','count')
                     ],
    'price_filtered': [
        ('cheapest', 'min'),
        ('max', 'max'),
        ('mean', 'mean')
    ]
}).rename(columns={'station_name': 'Stations','price_filtered':'Price'}).sort_values(by=('Stations', 'counts'), ascending=False)
)

sy[('Price', 'mean')] = sy[('Price', 'mean')].round(1)

##print(sy)

# 1. Flatten the columns
sy.columns = ['_'.join(col).strip() for col in sy.columns.values]

# 2. Round the now-flattened column
##sy['Price_mean'] = sy['Price_mean'].round(1)

# 3. Sort is now much cleaner (no tuples needed)
sy = sy.sort_values(by='Stations_counts', ascending=False)


##print(sy)


# Access any fuel type like:
# dfs['U91'], dfs['P98'], etc.
##print(dfs['P95'].columns)
# Check record counts per fuel type
##print(df.groupby('fuel_type').size())

In [ ]:
df = pd.DataFrame({
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'attr_1': [1, 2, 5, 3, 5, 4, 3],
    'attr_2': [3, 4, 5, 1, 5, 4, 3],              
    'attr_3': [2, 3, 5, 3, 5, 2, 3],
    
})

In [ ]:
import pandas as pd

import json

df = pd.read_csv('V1_TEST_20250312_survey_data_5000.csv')

print(df.info())

In [ ]:
df = pd.DataFrame({
    'Id':[4,3,2,1,5,6,7],
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'email': ['test1@test.com','test2@test.com','test1@test.com','test3@test.com','test2@test.com','test1@test.com','test2@test.com'],
    'survey_date': ['2025-01-01', '2023-01-01', '2024-01-01', '2026-01-01', '2022-01-01', '2025-11-01', '2024-10-01'],
    'attr_2': [3, 4, 5, 1, 5, 4, 3],              
    'attr_3': [2, 3, 5, 3, 5, 2, 3],
    
})

In [184]:
import pandas as pd

df = pd.DataFrame({
    'Id': [4, 3, 2, 1, 5, 6, 7],
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C'],
    'email': ['test1@test.com','test2@test.com','test1@test.com','test3@test.com','test2@test.com','test1@test.com','test2@test.com'],
    'survey_date': ['2025-01-01', '2023-01-01', '2024-01-01', '2026-01-01', '2022-01-01', '2025-11-01', '2024-10-01'],
    'attr_2': [3, 4, 5, 1, 5, 4, 3],
    'attr_3': [2, 3, 5, 3, 5, 2, 3],
})

df['survey_date'] = pd.to_datetime(df['survey_date'])

df_sorted = df.sort_values(['email', 'survey_date'], ascending=[True, True]).reset_index(drop=True)

##print(df_sorted)

df_sorted['survey_attempt'] = df_sorted.groupby('email').cumcount() + 1

print(df_sorted)


# Option 1: filter where survey_attempt > 1
df_duplicates = df_sorted[df_sorted['survey_attempt'] > 1]

# Option 2: get emails that have more than 1 attempt, then filter all their rows
emails_with_multiple = df_sorted.groupby('email')['survey_attempt'].max()
print(emails_with_multiple)
emails_with_multiple = emails_with_multiple[emails_with_multiple > 1].index
print(emails_with_multiple)
df_duplicates = df_sorted[df_sorted['email'].isin(emails_with_multiple)]


##print(df_duplicates)

# Get IDs where survey_attempt > 1
duplicate_ids = df_sorted[df_sorted['survey_attempt'] > 1]['Id'].tolist()
print(duplicate_ids)

# Exclude those IDs from the dataframe
df_clean = df_sorted[~df_sorted['Id'].isin(duplicate_ids)]
print(df_clean)

   Id Team           email survey_date  attr_2  attr_3  survey_attempt
0   2    A  test1@test.com  2024-01-01       5       5               1
1   4    A  test1@test.com  2025-01-01       3       2               2
2   6    B  test1@test.com  2025-11-01       4       2               3
3   5    B  test2@test.com  2022-01-01       5       5               1
4   3    A  test2@test.com  2023-01-01       4       3               2
5   7    C  test2@test.com  2024-10-01       3       3               3
6   1    B  test3@test.com  2026-01-01       1       3               1
email
test1@test.com    3
test2@test.com    3
test3@test.com    1
Name: survey_attempt, dtype: int64
Index(['test1@test.com', 'test2@test.com'], dtype='object', name='email')
[4, 6, 3, 7]
   Id Team           email survey_date  attr_2  attr_3  survey_attempt
0   2    A  test1@test.com  2024-01-01       5       5               1
3   5    B  test2@test.com  2022-01-01       5       5               1
6   1    B  test3@test.com  202

In [190]:
import pandas as pd

df = pd.DataFrame({
    'Id': [4, 3, 2, 1, 5, 6, 7,8],
    'Team': ['A', 'A', 'A', 'B', 'B', 'B', 'C','C'],
    'email': ['test1@test.com','test2@test.com','test1@test.com','test3@test.com','test2@test.com','test1@test.com','test2@test.com','test3@test.com'],
    'survey_date': ['2025-01-01', '2023-01-01', '2024-01-01', '2026-01-01', '2022-01-01', '2025-11-01', '2024-10-01','2026-01-10'],
    'attr_2': [3, 4, 5, 1, 5, 4, 3, 5],
    'attr_3': [2, 3, 5, 3, 5, 2, 3, 2],
})

df['survey_date'] = pd.to_datetime(df['survey_date'])
df_sorted = df.sort_values(['email', 'survey_date'], ascending=[True, True]).reset_index(drop=True)
df_sorted['survey_attempt'] = df_sorted.groupby('email').cumcount() + 1

# Get the first survey_date (attempt=1) per email as anchor
first_date = (df_sorted[df_sorted['survey_attempt'] == 1]
              .set_index('email')['survey_date']
              .rename('first_survey_date'))

df_sorted = df_sorted.join(first_date, on='email')

# Calculate months difference from attempt 1
df_sorted['months_from_attempt_1'] = (
    (df_sorted['survey_date'].dt.year  - df_sorted['first_survey_date'].dt.year) * 12 +
    (df_sorted['survey_date'].dt.month - df_sorted['first_survey_date'].dt.month)
)

# Days difference from attempt 1
df_sorted['days_from_attempt_1'] = (df_sorted['survey_date'] - df_sorted['first_survey_date']).dt.days

print(df_sorted)

   Id Team           email survey_date  attr_2  attr_3  survey_attempt  \
0   2    A  test1@test.com  2024-01-01       5       5               1   
1   4    A  test1@test.com  2025-01-01       3       2               2   
2   6    B  test1@test.com  2025-11-01       4       2               3   
3   5    B  test2@test.com  2022-01-01       5       5               1   
4   3    A  test2@test.com  2023-01-01       4       3               2   
5   7    C  test2@test.com  2024-10-01       3       3               3   
6   1    B  test3@test.com  2026-01-01       1       3               1   
7   8    C  test3@test.com  2026-01-10       5       2               2   

  first_survey_date  months_from_attempt_1  days_from_attempt_1  
0        2024-01-01                      0                    0  
1        2024-01-01                     12                  366  
2        2024-01-01                     22                  670  
3        2022-01-01                      0                    0  
4  